# Part I - Full Fine-Tuning Baseline

This notebook is a Colab/Jupyter wrapper around the Python scripts in this folder. It trains an adapted XLM-RoBERTa causal language model baseline on `flytech/python-codes-25k`.

## Requirements Covered

| Requirement | Value |
| --- | --- |
| Base model | `FacebookAI/xlm-roberta-base` by default; `FacebookAI/xlm-roberta-large` optional |
| LM head | `AutoModelForCausalLM` with decoder adaptation |
| Dataset | `flytech/python-codes-25k` |
| Batch size | `4` |
| Gradient accumulation | `1` |
| Epochs | `2` |
| Learning rate | `5e-5` |
| Optimizer | `adamw_torch` |
| Mixed precision | BF16 when available, fp16 fallback on T4 unless strict BF16 is requested |
| Gradient checkpointing | enabled |
| Scheduler | cosine with warmup |
| Logging | Weights & Biases |

In [1]:
# Run this in Colab/Kaggle before training.
!pip install -q torch transformers datasets accelerate peft trl bitsandbytes wandb


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Optional, but recommended for the assignment analysis.
# import wandb
# wandb.login()

In [3]:
import config

print('Model:', config.BASE_MODEL_ID)
print('Dataset:', config.DATASET_NAME)
print('Batch size:', config.PER_DEVICE_TRAIN_BATCH_SIZE)
print('Gradient accumulation:', config.GRADIENT_ACCUMULATION_STEPS)
print('Epochs:', config.NUM_TRAIN_EPOCHS)
print('Learning rate:', config.LEARNING_RATE)
print('Optimizer:', config.OPTIMIZER)
print('Gradient checkpointing:', config.GRADIENT_CHECKPOINTING)

Model: FacebookAI/xlm-roberta-base
Dataset: flytech/python-codes-25k
Batch size: 4
Gradient accumulation: 1
Epochs: 2
Learning rate: 5e-05
Optimizer: adamw_torch
Gradient checkpointing: True


## Smoke Test

Run this first to check that the environment, imports, dataset loading, and training loop work.

In [4]:
!python train.py --sample_size 32 --no_wandb

c:\Users\Wind\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Wind\.cache\huggingface\hub\models--FacebookAI--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Wind\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\

## Full Assignment Run

Run this for the actual Part I experiment. It uses the full dataset by default.

In [5]:
!python train.py


Formatting code-generation examples: 100%|██████████| 49626/49626 [00:01<00:00, 29448.35 examples/s]

Tokenizing FFT dataset: 100%|██████████| 49626/49626 [00:03<00:00, 14824.72 examples/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 9093.39it/s]
[transformers] XLMRobertaForCausalLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Traceback (most recent call last):
  File "d:\Term 8\NLP\NLP-Projects\assigment_4\part1_fft_roberta\train.py", line 120, in <module>
    main()
    ~~~~^^
  File "d:\Term 8\NLP\NLP-Projects\assigment_4\part1_fft_roberta\train.py", line 87, in main
    training_args = TrainingArguments(
        output_dir=args.output_dir,
    ...<14 lines>...
   

The final model is saved to `./fft_roberta_model`, and training checkpoints/logs are saved under `./results_fft_roberta`.